We had gotten feedback to try using an elastic net regression model.

# Elastic Net for Risk

In [38]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt

In [39]:
DATA_FILE = "../../data/processed/cardio_onc_prostate_06_broad_clean.csv"
OUT_DIR = "Results/ElasticNet"
SEED = 42

os.makedirs(OUT_DIR, exist_ok = True)

In [41]:
df = pd.read_csv(DATA_FILE)

df["at_risk"] = (
    (df["bp_meds_post_binary"] == 1) |
    (df["lipid_meds_post_binary"] == 1) |
    (df["dm_meds_post_binary"] == 1)
).astype(int)

In [5]:
continuous_features = [
    "bmi", "age", "sbp", "dbp", "ascvd_10yr"
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features = [
    "ethnicity_enc", "specific_nht_used_enc",
    "adt_agent_enc", "prescribing_provider_enc",
]

all_features = continuous_features + binary_features + encoded_features

In [6]:
X_df = df[all_features].copy().astype(float)
y = df["at_risk"].values.astype(int)
X_arr = X_df.values

## Preprocess

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cont", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), continuous_features),

        ("binary", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
        ]), binary_features),

        ("encoded", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
        ]), encoded_features),
    ],
    remainder="drop",
)

X_processed = preprocessor.fit_transform(X_df)

## Model Fit

In [8]:
model = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.1, 0.5, 0.9],
    Cs=np.logspace(-4, 2, 50),
    cv=5,
    scoring='roc_auc',
    max_iter=10000,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_processed, y)

LogisticRegressionCV(Cs=array([1.00000000e-04, 1.32571137e-04, 1.75751062e-04, 2.32995181e-04,
       3.08884360e-04, 4.09491506e-04, 5.42867544e-04, 7.19685673e-04,
       9.54095476e-04, 1.26485522e-03, 1.67683294e-03, 2.22299648e-03,
       2.94705170e-03, 3.90693994e-03, 5.17947468e-03, 6.86648845e-03,
       9.10298178e-03, 1.20679264e-02, 1.59985872e-02, 2.12095089e-02,
       2.81176870e-02, 3.72...
       8.28642773e-01, 1.09854114e+00, 1.45634848e+00, 1.93069773e+00,
       2.55954792e+00, 3.39322177e+00, 4.49843267e+00, 5.96362332e+00,
       7.90604321e+00, 1.04811313e+01, 1.38949549e+01, 1.84206997e+01,
       2.44205309e+01, 3.23745754e+01, 4.29193426e+01, 5.68986603e+01,
       7.54312006e+01, 1.00000000e+02]),
                     class_weight='balanced', cv=5, l1_ratios=[0.1, 0.5, 0.9],
                     max_iter=10000, penalty='elasticnet', random_state=42,
                     scoring='roc_auc', solver='saga')

## Extract Associations

In [9]:
# Get feature names from the preprocessor
onehot_features = preprocessor.named_transformers_['encoded']['onehot'].get_feature_names_out(encoded_features).tolist()
all_processed_features = continuous_features + binary_features + onehot_features

# Build the dataframe
coef_df = pd.DataFrame({
    'feature':    all_processed_features,
    'log_odds':   model.coef_[0],
    'odds_ratio': np.exp(model.coef_[0])
}).query('log_odds != 0').sort_values('log_odds', key=abs, ascending=False)

## Errors & P-values

In [10]:
X_processed_df = pd.DataFrame(X_processed, columns=all_processed_features)

selected = coef_df['feature'].tolist()
X_sel = sm.add_constant(X_processed_df[selected])

logit_model = sm.Logit(y, X_sel).fit()
print(logit_model.summary())

odds_ratios = pd.DataFrame({
    'OR':     np.exp(logit_model.params),
    'CI_low': np.exp(logit_model.conf_int()[0]),
    'CI_high':np.exp(logit_model.conf_int()[1]),
    'p_value':logit_model.pvalues
})

Optimization terminated successfully.
         Current function value: 0.549146
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  239
Model:                          Logit   Df Residuals:                      233
Method:                           MLE   Df Model:                            5
Date:                Mon, 18 May 2026   Pseudo R-squ.:                  0.1346
Time:                        08:03:15   Log-Likelihood:                -131.25
converged:                       True   LL-Null:                       -151.66
Covariance Type:            nonrobust   LLR p-value:                 1.016e-07
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -1.3700      0.323     -4.240      0.000      -2.003      -0.737
dm

## Conclusion

Lipid panel checked, non-insulin diabetes medication, and prior BP meds have the strongest association with risk, with p-values less than 0.05.

Health history of smoking is suggestive but not significant.

# NHT Choice (Abiraterone vs Darolutamide)

Committed changes for using all NHTs but didn't find any significant results, so refitting with just Abiraterone and Darolutamide.

In [42]:
df = pd.read_csv(DATA_FILE)
df_2class = df[df["specific_nht_used"].isin(["Abiraterone", "Darolutamide"])].copy()
df_2class["nht_binary"] = (df_2class["specific_nht_used"] == "Darolutamide").astype(int)

print(df_2class["nht_binary"].value_counts())

nht_binary
0    104
1     91
Name: count, dtype: int64


In [43]:
continuous_features = [
    "bmi", "age", "sbp", "dbp", "ascvd_10yr"
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features_new = [
    "ethnicity_enc",
    "adt_agent_enc", "prescribing_provider_enc",
]

all_features = continuous_features + binary_features + encoded_features_new

In [44]:
X_df = df_2class[all_features].copy().astype(float)
y = df_2class["nht_binary"].values.astype(int)

preprocessor_new = ColumnTransformer(
    transformers=[
        ("cont", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), continuous_features),
        ("binary", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
        ]), binary_features),
        ("encoded", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
        ]), encoded_features_new),
    ],
    remainder="drop",
)

X_processed = preprocessor_new.fit_transform(X_df)

In [45]:
model = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.1, 0.5, 0.9, 1.0],
    Cs=np.logspace(-2, 4, 50),
    cv=5,
    scoring='neg_log_loss',
    max_iter=10000,
    random_state=42,
    n_jobs=-1
)

model.fit(X_processed, y)

LogisticRegressionCV(Cs=array([1.00000000e-02, 1.32571137e-02, 1.75751062e-02, 2.32995181e-02,
       3.08884360e-02, 4.09491506e-02, 5.42867544e-02, 7.19685673e-02,
       9.54095476e-02, 1.26485522e-01, 1.67683294e-01, 2.22299648e-01,
       2.94705170e-01, 3.90693994e-01, 5.17947468e-01, 6.86648845e-01,
       9.10298178e-01, 1.20679264e+00, 1.59985872e+00, 2.12095089e+00,
       2.81176870e+00, 3.72...
       8.28642773e+01, 1.09854114e+02, 1.45634848e+02, 1.93069773e+02,
       2.55954792e+02, 3.39322177e+02, 4.49843267e+02, 5.96362332e+02,
       7.90604321e+02, 1.04811313e+03, 1.38949549e+03, 1.84206997e+03,
       2.44205309e+03, 3.23745754e+03, 4.29193426e+03, 5.68986603e+03,
       7.54312006e+03, 1.00000000e+04]),
                     cv=5, l1_ratios=[0.1, 0.5, 0.9, 1.0], max_iter=10000,
                     n_jobs=-1, penalty='elasticnet', random_state=42,
                     scoring='neg_log_loss', solver='saga')

In [46]:
print(f"Optimal C: {model.C_[0]:.6f}")
print(f"Optimal l1_ratio: {model.l1_ratio_[0]:.2f}")

Optimal C: 0.126486
Optimal l1_ratio: 0.90


In [47]:
onehot_features_new = preprocessor_new.named_transformers_['encoded']['onehot'].get_feature_names_out(encoded_features_new).tolist()
all_processed_features_new = continuous_features + binary_features + onehot_features_new

In [48]:
coef_df = pd.DataFrame({
    'feature':    all_processed_features_new,
    'log_odds':   model.coef_[0],
    'odds_ratio': np.exp(model.coef_[0])
}).query('log_odds != 0').sort_values('log_odds', key=abs, ascending=False)

print(f"\nFeatures selected: {len(coef_df)}")
print(coef_df)


Features selected: 3
      feature  log_odds  odds_ratio
1         age  0.444193    1.559231
4  ascvd_10yr -0.111834    0.894192
0         bmi -0.017805    0.982353


In [49]:
X_processed_df = pd.DataFrame(X_processed, columns=all_processed_features_new)
selected = coef_df['feature'].tolist()
X_sel = sm.add_constant(X_processed_df[selected])

logit_model = sm.Logit(y, X_sel).fit(maxiter=200)
print(logit_model.summary())

odds_ratios = pd.DataFrame({
    'OR':     np.exp(logit_model.params),
    'CI_low': np.exp(logit_model.conf_int()[0]),
    'CI_high':np.exp(logit_model.conf_int()[1]),
    'p_value': logit_model.pvalues
}).drop('const').sort_values('OR', ascending=False)

print(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.636307
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  195
Model:                          Logit   Df Residuals:                      191
Method:                           MLE   Df Model:                            3
Date:                Wed, 20 May 2026   Pseudo R-squ.:                 0.07905
Time:                        11:14:30   Log-Likelihood:                -124.08
converged:                       True   LL-Null:                       -134.73
Covariance Type:            nonrobust   LLR p-value:                 9.119e-05
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1455      0.152     -0.958      0.338      -0.443       0.152
age            0.6238      0.

## Results - NHT Choice

Age is the strongest finding. Older patients had 87% higher odds of receiving Darolutamide vs Abiraterone.

Intercept is non-significant, meaning at average values of all features there is no strong baseline preference for either drug.

# ADT Choice

In [87]:
df = pd.read_csv(DATA_FILE)
df = df[df["adt_agent"].notna()].copy()
print(df["adt_agent"].value_counts())

adt_agent
Orgovyx         83
Lupron          66
Bicalutamide    27
Firmagon        19
Name: count, dtype: int64


In [88]:
continuous_features = [
    "bmi", "age", "sbp", "dbp", "ascvd_10yr"
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features_new = [
    "ethnicity_enc",
    "specific_nht_used_enc", "prescribing_provider_enc",
]

all_features_adt = continuous_features + binary_features + encoded_features_new

In [89]:
# Check class balance
print(df["adt_agent"].value_counts())

# Check events per feature estimate
n_classes = df["adt_agent"].nunique()
min_class = df["adt_agent"].value_counts().min()
print(f"\nSmallest class: {min_class}")
print(f"Features before encoding: {len(all_features_adt)}")
print(f"Estimated features after encoding: ~{len(continuous_features) + len(binary_features) + 10}")  
print(f"Events per feature (smallest class): {min_class / (len(continuous_features) + len(binary_features) + 10):.1f}")

adt_agent
Orgovyx         83
Lupron          66
Bicalutamide    27
Firmagon        19
Name: count, dtype: int64

Smallest class: 19
Features before encoding: 41
Estimated features after encoding: ~48
Events per feature (smallest class): 0.4


In [91]:
preprocessor_new = ColumnTransformer(
    transformers=[
        ("cont", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), continuous_features),
        ("binary", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
        ]), binary_features),
        ("encoded", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
        ]), encoded_features_new),
    ],
    remainder="drop",
)

X_processed = preprocessor_new.fit_transform(X_df)

In [92]:
model = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    multi_class='multinomial',
    l1_ratios=[0.1, 0.5, 0.7, 0.9, 1.0],
    Cs=np.logspace(-2, 4, 50),
    cv=5,
    scoring='neg_log_loss',
    max_iter=10000,
    random_state=42
)
model.fit(X_processed, y)

c:\Users\Trian\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1905: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegressionCV(Cs=array([1.00000000e-02, 1.32571137e-02, 1.75751062e-02, 2.32995181e-02,
       3.08884360e-02, 4.09491506e-02, 5.42867544e-02, 7.19685673e-02,
       9.54095476e-02, 1.26485522e-01, 1.67683294e-01, 2.22299648e-01,
       2.94705170e-01, 3.90693994e-01, 5.17947468e-01, 6.86648845e-01,
       9.10298178e-01, 1.20679264e+00, 1.59985872e+00, 2.12095089e+00,
       2.81176870e+00, 3.72...
       2.55954792e+02, 3.39322177e+02, 4.49843267e+02, 5.96362332e+02,
       7.90604321e+02, 1.04811313e+03, 1.38949549e+03, 1.84206997e+03,
       2.44205309e+03, 3.23745754e+03, 4.29193426e+03, 5.68986603e+03,
       7.54312006e+03, 1.00000000e+04]),
                     cv=5, l1_ratios=[0.1, 0.5, 0.7, 0.9, 1.0], max_iter=10000,
                     multi_class='multinomial', penalty='elasticnet',
                     random_state=42, scoring='neg_log_loss', solver='saga')

In [93]:
print(f"Optimal C: {model.C_[0]:.6f}")
print(f"Optimal l1_ratio: {model.l1_ratio_[0]:.2f}")

Optimal C: 0.013257
Optimal l1_ratio: 1.00


In [95]:
onehot_features_new = preprocessor_new.named_transformers_['encoded']['onehot'].get_feature_names_out(encoded_features_new).tolist()
all_processed_features_new = continuous_features + binary_features + onehot_features_new

coef_matrix = pd.DataFrame(
    model.coef_,
    columns=all_processed_features_new
)

selected = coef_matrix.columns[(coef_matrix != 0).any(axis=0)].tolist()
print(f"Features selected: {len(selected)}")
print(selected)

X_processed_df = pd.DataFrame(X_processed, columns=all_processed_features_new)
X_sel = sm.add_constant(X_processed_df[selected])

mnlogit_model = sm.MNLogit(y, X_sel).fit()
print(mnlogit_model.summary())

Features selected: 0
[]
Optimization terminated successfully.
         Current function value: 1.230882
         Iterations 5
                          MNLogit Regression Results                          
Dep. Variable:                      y   No. Observations:                  195
Model:                        MNLogit   Df Residuals:                      192
Method:                           MLE   Df Model:                            0
Date:                Mon, 18 May 2026   Pseudo R-squ.:               6.518e-11
Time:                        19:59:35   Log-Likelihood:                -240.02
converged:                       True   LL-Null:                       -240.02
Covariance Type:            nonrobust   LLR p-value:                       nan
       y=1       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.3514      0.299     -1.173      0.241      -0.938       0.236
-----

In [12]:
df['adt_agent'].value_counts()

adt_agent
Orgovyx         83
Lupron          66
Bicalutamide    27
Firmagon        19
Name: count, dtype: int64

## Trying Just Orgovyx and Bicalutamide

In [66]:
df = pd.read_csv(DATA_FILE)
df_2class = df[df["adt_agent"].isin(["Orgovyx", "Bicalutamide"])].copy()
df_2class["adt_binary"] = (df_2class["adt_agent"] == "Bicalutamide").astype(int)

print(df_2class["adt_binary"].value_counts())

adt_binary
0    83
1    27
Name: count, dtype: int64


In [67]:
continuous_features = [
    "bmi", "age", "sbp", "dbp"
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features_new = [
    "ethnicity_enc",
    "specific_nht_used_enc", "prescribing_provider_enc",
]

all_features = continuous_features + binary_features + encoded_features_new

In [68]:
X_df = df_2class[all_features].copy().astype(float)
y = df_2class["adt_binary"].values.astype(int)

preprocessor_new = ColumnTransformer(
    transformers=[
        ("cont", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), continuous_features),
        ("binary", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
        ]), binary_features),
        ("encoded", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
        ]), encoded_features_new),
    ],
    remainder="drop",
)

X_processed = preprocessor_new.fit_transform(X_df)

In [69]:
model = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.1, 0.5, 0.9, 1.0],
    Cs=np.logspace(-2, 4, 50),
    cv=5,
    scoring='neg_log_loss',
    max_iter=10000,
    random_state=42,
    n_jobs=-1
)

model.fit(X_processed, y)

LogisticRegressionCV(Cs=array([1.00000000e-02, 1.32571137e-02, 1.75751062e-02, 2.32995181e-02,
       3.08884360e-02, 4.09491506e-02, 5.42867544e-02, 7.19685673e-02,
       9.54095476e-02, 1.26485522e-01, 1.67683294e-01, 2.22299648e-01,
       2.94705170e-01, 3.90693994e-01, 5.17947468e-01, 6.86648845e-01,
       9.10298178e-01, 1.20679264e+00, 1.59985872e+00, 2.12095089e+00,
       2.81176870e+00, 3.72...
       8.28642773e+01, 1.09854114e+02, 1.45634848e+02, 1.93069773e+02,
       2.55954792e+02, 3.39322177e+02, 4.49843267e+02, 5.96362332e+02,
       7.90604321e+02, 1.04811313e+03, 1.38949549e+03, 1.84206997e+03,
       2.44205309e+03, 3.23745754e+03, 4.29193426e+03, 5.68986603e+03,
       7.54312006e+03, 1.00000000e+04]),
                     cv=5, l1_ratios=[0.1, 0.5, 0.9, 1.0], max_iter=10000,
                     n_jobs=-1, penalty='elasticnet', random_state=42,
                     scoring='neg_log_loss', solver='saga')

In [70]:
print(f"Optimal C: {model.C_[0]:.6f}")
print(f"Optimal l1_ratio: {model.l1_ratio_[0]:.2f}")

Optimal C: 0.095410
Optimal l1_ratio: 0.10


In [71]:
onehot_features_new = preprocessor_new.named_transformers_['encoded']['onehot'].get_feature_names_out(encoded_features_new).tolist()
all_processed_features_new = continuous_features + binary_features + onehot_features_new

In [72]:
coef_df = pd.DataFrame({
    'feature':    all_processed_features_new,
    'log_odds':   model.coef_[0],
    'odds_ratio': np.exp(model.coef_[0])
}).query('log_odds != 0').sort_values('log_odds', key=abs, ascending=False)

print(f"\nFeatures selected: {len(coef_df)}")
print(coef_df)


Features selected: 27
                          feature  log_odds  odds_ratio
2                             sbp  0.268396    1.307864
7                   bp_meds_prior  0.246763    1.279876
30                     cards_post  0.197252    1.218051
24                  dm_noninsulin -0.188053    0.828571
28                        asa_use  0.175795    1.192194
16                         hx_cad  0.169035    1.184161
35                       ecg_done -0.151058    0.859798
42      specific_nht_used_enc_2.0 -0.150438    0.860331
73  prescribing_provider_enc_42.0  0.134409    1.143861
23                         hx_dm2  0.127364    1.135830
34                   echo_ordered  0.121145    1.128788
41      specific_nht_used_enc_1.0  0.115783    1.122752
38              ethnicity_enc_2.0 -0.106863    0.898649
26                    a1c_checked  0.106190    1.112033
15            lipid_panel_checked  0.105966    1.111784
1                             age  0.100602    1.105837
17                    hx_

In [73]:
X_processed_df = pd.DataFrame(X_processed, columns=all_processed_features_new)
selected = coef_df['feature'].tolist()
X_sel = sm.add_constant(X_processed_df[selected])

logit_model = sm.Logit(y, X_sel).fit()
print(logit_model.summary())

odds_ratios = pd.DataFrame({
    'OR':     np.exp(logit_model.params),
    'CI_low': np.exp(logit_model.conf_int()[0]),
    'CI_high':np.exp(logit_model.conf_int()[1]),
    'p_value': logit_model.pvalues
}).drop('const').sort_values('OR', ascending=False)

print(odds_ratios)

         Current function value: 0.272979
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  110
Model:                          Logit   Df Residuals:                       82
Method:                           MLE   Df Model:                           27
Date:                Wed, 20 May 2026   Pseudo R-squ.:                  0.5102
Time:                        16:21:21   Log-Likelihood:                -30.028
converged:                      False   LL-Null:                       -61.301
Covariance Type:            nonrobust   LLR p-value:                 0.0001211
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const                            -1.9423      0.941     -2.063      0.039      -3.787      -0.097
sbp     

c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Conclusion

Potentially higher SBP is associated with lower Orgovyx use.